In [1]:
import torch
import torchvision
import os

In [2]:
from torchvision import datasets, transforms

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load dataset
dataset = datasets.ImageFolder(root='dataset', transform=transform)

# Check classes
print(dataset.classes)

FileNotFoundError: Found no valid file for the classes Crack, No_Crack. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [3]:
import os

print("Crack:", len(os.listdir("dataset/Crack")))
print("No_Crack:", len(os.listdir("dataset/No_Crack")))

print("Crack sample:", os.listdir("dataset/Crack")[:5])
print("No_Crack sample:", os.listdir("dataset/No_Crack")[:5])

Crack: 20000
No_Crack: 12907
Crack sample: ['00001.jpg', '00002.jpg', '00003.jpg', '00004.jpg', '00005.jpg']
No_Crack sample: ['07094.jpg', '07095.jpg', '07096.jpg', '07097.jpg', '07098.jpg']


In [4]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(root="dataset", transform=transform)

print(dataset.classes)
print("Total images:", len(dataset))

['Crack', 'No_Crack']
Total images: 32907


In [5]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 26325
Test: 6582


In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print("DataLoader ready")

DataLoader ready


In [7]:
import torch.nn as nn
import torchvision.models as models

model = models.mobilenet_v2(pretrained=True)

# Modify final layer
model.classifier[1] = nn.Linear(model.last_channel, 2)

print(model)

c:\AI\CrackDetection\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\AI\CrackDetection\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\smbab/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100.0%


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [8]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Ready to train")

Ready to train


In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 1   # keep 1 for now (faster)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Batch {batch_idx}, Loss: {loss.item():.4f}")
    
    print(f"Epoch {epoch+1} completed, Total Loss: {total_loss:.4f}")

Batch 0, Loss: 0.0022
Batch 100, Loss: 0.0003
Batch 200, Loss: 0.0002
Batch 300, Loss: 0.0006
Batch 400, Loss: 0.0005
Batch 500, Loss: 0.0009
Batch 600, Loss: 0.0007
Batch 700, Loss: 0.3891
Batch 800, Loss: 0.0012
Epoch 1 completed, Total Loss: 16.2735


In [11]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 99.67%


In [12]:
torch.save(model.state_dict(), "crack_detection_model.pth")
print("Model saved successfully")

Model saved successfully
